# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library. We follow best practices for referencing dataset entities by their `@id` fields throughout.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Show other high-level metadata
print('Version:', metadata.version)
print('Authors:', metadata.author)
print('Published:', metadata.date_published)
print('Keywords:', metadata.keywords)

## 2. Data Overview
Review available record sets, fields, and their `@id` entries.

We list all Record Sets, their names, and included Fields using their `@id` values for clarity. This will help us when extracting and analyzing data.

In [ ]:
# List RecordSets and Fields by @id
for record_set in metadata.record_sets:
    print(f"RecordSet: {record_set.name} (id: {record_set.id})")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print("  Columns:")
    for column in record_set.columns:
        print(f"    - {column.name} (@id: {column.id})")
    print('')

Let's take a peek at the records from a selected record set. Substitute your desired record set's `@id` below.

In [ ]:
# Example: Print the first 2 records from the main protein data record set
# Find the appropriate RecordSet @id (e.g., 'https://api.app.sen.science/frontiers/7154140/XXXX')
main_recordset_id = metadata.record_sets[0].id  # Use the first record set as example
print(f"Sample records from RecordSet: {main_recordset_id}")
for i, record in enumerate(dataset.records(record_set=main_recordset_id)):
    print(record)
    if i >= 1:
        break

## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames for analysis. Use the `@id` of each record set.

The `@id` can be obtained from the previous cell.

In [ ]:
# Extract all record sets into DataFrames, referencing by @id
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  Example rows:\n", df.head(2), "\n")
    else:
        print(f"  No records found in RecordSet: {record_set_id}\n")

For demonstration in the following cells, we'll use the first available record set as the main DataFrame. Update the variable `target_record_set_id` if you wish to use a different one.

In [ ]:
# Choose which record set to analyze (by @id)
target_record_set_id = record_set_ids[0] if record_set_ids else None
main_df = dataframes[target_record_set_id] if target_record_set_id else None
if main_df is not None:
    print(f"Main DataFrame for RecordSet {target_record_set_id} is loaded. Columns:")
    print(main_df.columns.tolist())
    display(main_df.head())
else:
    print("No data found in any record set.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data.

We'll select a numeric field (by `@id`) to demonstrate filtering, normalization, and grouping. Replace `example_numeric_field` and `group_field` with real `@id` values from your overview as needed.

In [ ]:
if main_df is not None and len(main_df.columns) > 0:
    # Set field IDs for demonstration
    # Inspect available columns to pick likely numeric and group fields
    print('Main DataFrame columns:', main_df.columns.tolist())
    # Example selections (update these using correct @id):
    numeric_field = None
    for col in main_df.columns:
        # Attempt to guess a numeric field (update to match the dataset field @id)
        if 'coverage' in col.lower() or 'count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower():
            if pd.api.types.is_numeric_dtype(main_df[col]):
                numeric_field = col
                break
    if not numeric_field:
        # If not found, just select the first float/int column
        for col in main_df.columns:
            if pd.api.types.is_numeric_dtype(main_df[col]):
                numeric_field = col
                break
    print(f"Selected numeric field: {numeric_field}")

    # Now select a group field (e.g., 'accession' or 'Modification' by @id if present)
    group_field = None
    for col in main_df.columns:
        if 'group' in col.lower() or 'type' in col.lower() or 'class' in col.lower() or 'accession' in col.lower():
            group_field = col
            break
    if not group_field and len(main_df.columns) > 1:
        group_field = main_df.columns[1]
    print(f"Selected group field: {group_field}")

    # Set a threshold (use median as a default if available)
    threshold = None
    if numeric_field:
        threshold = main_df[numeric_field].median() if not main_df[numeric_field].isnull().all() else 0
        # Filter
        filtered_df = main_df[main_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} ({len(filtered_df)} rows):")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df = filtered_df.copy()
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        )
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by chosen field
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped (mean) {numeric_field} by {group_field}:")
            print(grouped_df.head())
else:
    print('No suitable DataFrame or columns for EDA.')

## 5. Visualization
Visualize a histogram of the selected numeric field and a boxplot grouped by the group field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and numeric_field is not None and numeric_field in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot if group_field is valid
    if group_field and group_field in main_df.columns:
        # Limit number of groups for clarity
        top_groups = main_df[group_field].value_counts().index[:10]
        plt.figure(figsize=(10,5))
        sns.boxplot(
            data=main_df[main_df[group_field].isin(top_groups)],
            x=group_field, y=numeric_field
        )
        plt.title(f"{numeric_field} by {group_field} (Top 10)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Not enough data for visualization.')

## 6. Conclusion
We demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library, referencing entities by their `@id`. We conducted initial EDA and visualized key numeric fields. For more advanced exploration, further domain knowledge and additional cleaning steps may be applied, tailored to the fields present in each record set.

*Notebook prepared using the [mlcroissant](https://github.com/mlcommons/croissant) Python library and the FAIR^2 dataset specification.*